<a href="https://colab.research.google.com/github/Mutaqie/Latihan-Machine-Learning/blob/main/Pertemuan%204/Latihan_4_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#1. Import Library dan Muat Data

In [3]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn import set_config

set_config(display="text")
import os
if not os.path.exists("Titanic-Dataset.csv"):
    from google.colab import files
    files.upload()   # pilih file Titanic-Dataset.csv

titanic = pd.read_csv("Titanic-Dataset.csv")
print("Ukuran data:", titanic.shape)
print("Kolom asli:", list(titanic.columns))

# Samakan nama kolom ke huruf kecil agar sesuai dengan kode di materi
titanic.columns = titanic.columns.str.lower()
titanic[["survived", "age", "fare", "sex", "embarked"]].head()

Ukuran data: (891, 12)
Kolom asli: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,survived,age,fare,sex,embarked
0,0,22.0,7.2500,male,S
1,1,38.0,71.2833,female,C
2,1,26.0,7.9250,female,S
3,1,35.0,53.1000,female,S
4,0,35.0,8.0500,male,S


In [4]:
print("Jumlah nilai kosong per kolom:")
print(titanic[["age", "fare", "sex", "embarked", "survived"]].isna().sum())
print("\nProporsi kelas target:")
print(titanic["survived"].value_counts(normalize=True).round(3))

Jumlah nilai kosong per kolom:
age         177
fare          0
sex           0
embarked      2
survived      0
dtype: int64

Proporsi kelas target:
survived
0    0.616
1    0.384
Name: proportion, dtype: float64


In [ ]:
# Langkah 1: Buat split terstratifikasi

In [5]:
X = titanic[["age", "fare", "sex", "embarked"]]
y = titanic["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("\nProporsi survived (train):", y_train.mean().round(4))
print("Proporsi survived (test) :", y_test.mean().round(4))

Train: (712, 4) | Test: (179, 4)

Proporsi survived (train): 0.3834
Proporsi survived (test) : 0.3855


In [ ]:
#Langkah 2: Definisikan kolom

In [7]:
fitur_numerik = ["age", "fare"]
fitur_kategorikal = ["sex", "embarked"]

print("Fitur numerik    :", fitur_numerik)
print("Fitur kategorikal:", fitur_kategorikal)
print("\nTipe data:")
print(X.dtypes)

Fitur numerik    : ['age', 'fare']
Fitur kategorikal: ['sex', 'embarked']

Tipe data:
age         float64
fare        float64
sex          object
embarked     object
dtype: object


In [ ]:
#Langkah 3: Bangun ColumnTransformer

In [8]:
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", num_pipe, ["age", "fare"]),
    ("cat", cat_pipe, ["sex", "embarked"])
])

pipe = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print("Pipeline berhasil dilatih.")
print("Akurasi test (semua fitur):", round(accuracy_score(y_test, y_pred), 4))

Pipeline berhasil dilatih.
Akurasi test (semua fitur): 0.7765


In [9]:
# Lihat fitur hasil transformasi
nama_fitur = pipe.named_steps["prep"].get_feature_names_out()
print("Fitur setelah preprocessing:")
for n in nama_fitur:
    print(" -", n)

# Bukti: statistik dipelajari dari TRAIN saja
imp_num = pipe.named_steps["prep"].named_transformers_["num"].named_steps["imputer"]
print("\nMedian yang dipelajari dari train (age, fare):", imp_num.statistics_.round(3))
imp_cat = pipe.named_steps["prep"].named_transformers_["cat"].named_steps["imputer"]
print("Modus yang dipelajari dari train (sex, embarked):", imp_cat.statistics_)

Fitur setelah preprocessing:
 - num__age
 - num__fare
 - cat__sex_female
 - cat__sex_male
 - cat__embarked_C
 - cat__embarked_Q
 - cat__embarked_S

Median yang dipelajari dari train (age, fare): [28.5   14.454]
Modus yang dipelajari dari train (sex, embarked): ['male' 'S']


In [ ]:
#Langkah 4: Bandingkan semua fitur vs fitur terpilih

In [10]:
#Uji chi-square (filter method) pada data TRAIN: fitur kategorikal vs survived
hasil_chi = []
for kolom in ["sex", "embarked"]:
    tabel = pd.crosstab(X_train[kolom].fillna(X_train[kolom].mode()[0]), y_train)
    chi2, p, dof, expected = chi2_contingency(tabel, correction=False)
    hasil_chi.append({"fitur": kolom, "chi2": round(chi2, 3), "df": dof, "p-value": p})
    print(f"Tabel observasi {kolom} vs survived:")
    print(tabel, "\n")

tabel_chi = pd.DataFrame(hasil_chi)
tabel_chi["keputusan (alpha=0,05)"] = np.where(tabel_chi["p-value"] < 0.05, "ada hubungan", "tidak signifikan")
tabel_chi

Tabel observasi sex vs survived:
survived    0    1
sex               
female     65  188
male      374   85 

Tabel observasi embarked vs survived:
survived    0    1
embarked          
C          61   78
Q          31   24
S         347  171 



,fitur,chi2,df,p-value,"keputusan (alpha=0,05)"
0,sex,214.731,1,1.275507e-48,ada hubungan
1,embarked,25.450,2,2.975141e-06,ada hubungan


In [11]:
#Korelasi fitur numerik (cek redundansi)
print("Korelasi age vs fare:", X_train[["age", "fare"]].corr().iloc[0, 1].round(3))
print("(Nilai kecil berarti kedua fitur tidak redundan)")

Korelasi age vs fare: 0.107
(Nilai kecil berarti kedua fitur tidak redundan)


In [12]:
#Skor filter (ANOVA F) pada fitur hasil transformasi
Xt = pipe.named_steps["prep"].transform(X_train)
selector_all = SelectKBest(score_func=f_classif, k="all").fit(Xt, y_train)
skor = (pd.DataFrame({"fitur": nama_fitur, "skor_F": selector_all.scores_.round(2)})
          .sort_values("skor_F", ascending=False).reset_index(drop=True))
skor

,fitur,skor_F
0,cat__sex_female,306.59
1,cat__sex_male,306.59
2,num__fare,58.31
3,cat__embarked_C,23.78
4,cat__embarked_S,23.55
5,num__age,3.78
6,cat__embarked_Q,0.71


In [13]:
#Bangun dua pipeline: SEMUA fitur vs fitur TERPILIH (SelectKBest k=4)
pipe_semua = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

pipe_terpilih = Pipeline([
    ("prep", preprocess),
    ("select", SelectKBest(score_func=f_classif, k=4)),
    ("model", LogisticRegression(max_iter=1000))
])

pipe_semua.fit(X_train, y_train)
pipe_terpilih.fit(X_train, y_train)

mask = pipe_terpilih.named_steps["select"].get_support()
fitur_terpilih = nama_fitur[mask]
fitur_dibuang = nama_fitur[~mask]
print("Fitur TERPILIH (k=4):")
for f in fitur_terpilih: print("  +", f)
print("\nFitur DIBUANG:")
for f in fitur_dibuang: print("  -", f)

Fitur TERPILIH (k=4):
  + num__fare
  + cat__sex_female
  + cat__sex_male
  + cat__embarked_C

Fitur DIBUANG:
  - num__age
  - cat__embarked_Q
  - cat__embarked_S


In [ ]:
#Langkah 5: Laporkan akurasi dan F1

In [14]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluasi(nama, model, jumlah_fitur):
    y_hat = model.predict(X_test)
    cv_f1 = cross_val_score(model, X_train, y_train, cv=skf, scoring="f1")
    cv_acc = cross_val_score(model, X_train, y_train, cv=skf, scoring="accuracy")
    return {
        "Skenario": nama,
        "Jumlah fitur": jumlah_fitur,
        "Akurasi (test)": round(accuracy_score(y_test, y_hat), 4),
        "F1 (test)": round(f1_score(y_test, y_hat), 4),
        "Akurasi (CV mean)": round(cv_acc.mean(), 4),
        "F1 (CV mean)": round(cv_f1.mean(), 4),
    }

tabel_metrik = pd.DataFrame([
    evaluasi("Semua fitur", pipe_semua, len(nama_fitur)),
    evaluasi("Fitur terpilih (SelectKBest k=4)", pipe_terpilih, 4),
])
tabel_metrik

print("=== Classification report: SEMUA fitur ===")
print(classification_report(y_test, pipe_semua.predict(X_test), target_names=["Tidak selamat", "Selamat"]))
print("=== Classification report: fitur TERPILIH ===")
print(classification_report(y_test, pipe_terpilih.predict(X_test), target_names=["Tidak selamat", "Selamat"]))

=== Classification report: SEMUA fitur ===
               precision    recall  f1-score   support

Tidak selamat       0.80      0.85      0.82       110
      Selamat       0.73      0.67      0.70        69

     accuracy                           0.78       179
    macro avg       0.77      0.76      0.76       179
 weighted avg       0.77      0.78      0.77       179

=== Classification report: fitur TERPILIH ===
               precision    recall  f1-score   support

Tidak selamat       0.80      0.85      0.82       110
      Selamat       0.73      0.67      0.70        69

     accuracy                           0.78       179
    macro avg       0.77      0.76      0.76       179
 weighted avg       0.77      0.78      0.77       179



In [ ]:
#Output akhir: Pipeline + daftar fitur + tabel metrik

In [15]:
print("PIPELINE (fitur terpilih):")
print(pipe_terpilih)
print("\nDAFTAR FITUR AKHIR YANG DIPAKAI MODEL:")
for i, f in enumerate(fitur_terpilih, 1):
    print(f"  {i}. {f}")
print("\nTABEL METRIK:")
print(tabel_metrik.to_string(index=False))

PIPELINE (fitur terpilih):
Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                               